# Open Duck Mini V2：**お盆なし歩行PPO 150M・7段階の動画復元**

**再学習なし・元の重みは変更しません。** Google Drive の `experiments/ppo_150m_v1/checkpoints/` に保存されたPPO重みから、**10M・30M・40M・60M・75M・100M・151Mに近い7段階**を同一条件で録画します。

| 比較する時期 | 正確な学習ステップ | 比較動画ラベル |
|---|---:|---|
| 約10M | 10,813,440 | `early_10.81M` |
| **約30M（追加）** | **32,440,320** | `early_32.44M` |
| 約40M | 43,253,760 | `earlymid_43.25M` |
| **約60M（追加）** | **64,880,640** | `middle_64.88M` |
| 約75M | 75,694,080 | `middle_75.69M` |
| **約100M（追加）** | **97,320,960** | `late_97.32M` |
| 約151M | 151,388,160 | `final_151.39M` |

**指定値ちょうどではなく、保存済みチェックポイントのうち最も近いもの**を使います。特に60M付近は64.88M、100M付近は97.32Mです。

録画条件は前の4段階版と同じ：**速度指令0.12m/s・8秒・seed 1300・15fps**。ノイズとランダム外力を無効にした比較用評価で、当時の学習時環境を完全再現した評価ではありません。

新しい出力フォルダは `OpenDuckRL/experiments/ppo_150m_v1/walk_video_replay_7stages/` です。**既存の4段階版フォルダとチェックポイントを変更しません。** 旧4本の比較条件をレポートで確認できる場合はファイルをそのまま再利用し、追加の3本を生成します。旧ファイルやレポートがなければ、その段階の動画も復元・録画します。

必須：元の `source/playground/open_duck_mini_v2/joystick.py`。観測・行動の次元が合わない場合は誤った再生を防ぐため停止します。GPUランタイム（L4/T4など）を推奨します。


## 1. Drive接続と保存済みチェックポイントの確認

In [1]:
from google.colab import drive
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/OpenDuckRL')
if not DRIVE.is_dir():
    drive.mount('/content/drive')

EXP = DRIVE / 'experiments/ppo_150m_v1'
SNAP = EXP / 'source'
SNAP_JOYSTICK = SNAP / 'playground/open_duck_mini_v2/joystick.py'
CKPT_ROOT = EXP / 'checkpoints'
CHECKPOINT_STEPS = [
    10_813_440, 32_440_320, 43_253_760, 64_880_640,
    75_694_080, 97_320_960, 151_388_160,
]

assert EXP.is_dir(), f'実験フォルダがありません: {EXP}'
assert CKPT_ROOT.is_dir(), CKPT_ROOT
for steps in CHECKPOINT_STEPS:
    matches = list(CKPT_ROOT.glob(f'*_{steps}'))
    assert len(matches) == 1 and matches[0].is_dir(), (steps, matches)
    assert any(matches[0].iterdir()), f'チェックポイントが空: {matches[0]}'
    print(f'{steps/1e6:>7.2f}M : {matches[0]}')

print('\n学習時ソース:', SNAP_JOYSTICK)
print('存在:', SNAP_JOYSTICK.is_file())
assert SNAP_JOYSTICK.is_file(), (
    '150M当時のjoystick.pyがありません。別の公開版で推測して復元するのは危険です。'
    'SOURCEフォルダの一覧を確認してください: ' + str(SNAP)
)
print('\n手順1 完了：7つの重みと学習時ソースを確認')

Mounted at /content/drive
  10.81M : /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints/2026_10_09_072121_10813440
  32.44M : /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints/2026_10_09_072729_32440320
  43.25M : /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints/2026_10_09_073016_43253760
  64.88M : /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints/2026_10_09_073550_64880640
  75.69M : /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints/2026_10_09_073836_75694080
  97.32M : /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints/2026_10_09_074410_97320960
 151.39M : /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints/2026_10_09_075804_151388160

学習時ソース: /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/source/playground/open_duck_mini_v2/joystick.py
存在: True

手順1 完了：7つの重みと学習時ソースを確認


## 2. 推論専用の作業ディレクトリを作成（オリジナルは変更しない）

Colab内に元Open Duck Playgroundのコードの**作業用コピー**を作り、150M当時に保存したソースを上書きして使用します。Drive上の学習ソースや重みは変更しません。

In [2]:
import shutil, subprocess, json, hashlib
WORK = Path('/content/duck_150m_walk_review')
BASE = Path('/content/Open_Duck_Playground')
REPO = 'https://github.com/apirrone/Open_Duck_Playground.git'

if not (WORK / 'playground/open_duck_mini_v2/joystick.py').is_file():
    if WORK.exists() and any(WORK.iterdir()):
        raise RuntimeError('専用WORKフォルダが不完全です。内容を確認してください: '+str(WORK))
    if (BASE / 'playground/open_duck_mini_v2/joystick.py').is_file():
        shutil.copytree(BASE, WORK, dirs_exist_ok=True,
            ignore=shutil.ignore_patterns('.git','.venv','__pycache__','*.mp4','checkpoints','.tmp'))
        print('既存Open_Duck_Playgroundのソースを専用WORKに複製')
    else:
        subprocess.run(['git','clone','--depth','1',REPO,str(WORK)],check=True)

# 150M当時の保存ソースを優先。公開リポジトリのみでは再現できない場合があります。
snapshot_pg = SNAP / 'playground'
assert snapshot_pg.is_dir(), snapshot_pg
shutil.copytree(snapshot_pg, WORK/'playground', dirs_exist_ok=True)
assert (WORK/'playground/open_duck_mini_v2/joystick.py').read_bytes() == SNAP_JOYSTICK.read_bytes()

for f in sorted(snapshot_pg.rglob('*.py')):
    print('学習時ソース適用:', f.relative_to(SNAP))
print('作業用ディレクトリ:',WORK)
print('元のDriveソース・XML・チェックポイントは未変更')

学習時ソース適用: playground/common/rewards.py
学習時ソース適用: playground/open_duck_mini_v2/custom_rewards.py
学習時ソース適用: playground/open_duck_mini_v2/joystick.py
学習時ソース適用: playground/open_duck_mini_v2/runner.py
作業用ディレクトリ: /content/duck_150m_walk_review
元のDriveソース・XML・チェックポイントは未変更


## 3. 復元専用Python環境を用意

既存ランタイムに十分な依存関係があれば再利用できますが、ここでは**分離したPython 3.12環境**を作ります。実際のGPUランタイムやパッケージ状態によってインストールに時間がかかる場合があります。

In [3]:
import os, sys, shutil, subprocess
VENV = Path('/content/duck_150m_walk_video_venv')
PYTHON = VENV / 'bin/python'

if not PYTHON.is_file():
    uv = shutil.which('uv')
    if uv is None:
        subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
        uv = shutil.which('uv')
    assert uv, 'uv のインストールを確認してください'
    subprocess.run([uv,'python','install','3.12'],check=True)
    subprocess.run([uv,'venv','--python','3.12',str(VENV)],check=True)

check_cmd = [str(PYTHON),'-c',
    'import jax, brax, mujoco, mujoco_playground, flax, orbax; from mujoco_playground.config import locomotion_params']
check = subprocess.run(check_cmd, capture_output=True, text=True)
if check.returncode:
    uv = shutil.which('uv')
    subprocess.run([uv,'pip','install','--python',str(PYTHON),
        'jax[cuda12]','playground','brax','mujoco','mujoco-mjx',
        'orbax-checkpoint','flax','ml-collections','etils','imageio[ffmpeg]',
        'pillow','mediapy','tensorboardX','jaxtyping'], check=True)

subprocess.run(check_cmd, check=True)
subprocess.run([str(PYTHON),'-c',
    'import jax, mujoco, brax; print("JAX:", jax.__version__); print("Devices:",jax.devices()); print("MuJoCo:",mujoco.__version__)'], check=True)
assert shutil.which('ffmpeg'), 'ffmpegが見つかりません'
print('推論用Python:',PYTHON)

推論用Python: /content/duck_150m_walk_video_venv/bin/python


## 4. 古いcollision importとの互換性を確認

旧版 `joystick.py` の `geoms_colliding` の読み込み先が新しいライブラリで変わったときだけ、**作業用コピーだけ**を修正します。学習時ソースやチェックポイントは変更しません。

In [4]:
from pathlib import Path
import shutil, subprocess
from hashlib import sha256

joystick_copy = WORK/'playground/open_duck_mini_v2/joystick.py'
probe = subprocess.run([str(PYTHON),'-c',
    'from mujoco_playground._src.collision import geoms_colliding'],
    capture_output=True, text=True)
if probe.returncode != 0:
    compat_candidates = [
        DRIVE/'delivery_ppo_stage2_150m/continuation_50m_source_snapshot/carry_compat.py',
        DRIVE/'delivery_ppo_stage2_150m/source/carry_compat.py',
    ]
    compat = next((p for p in compat_candidates if p.is_file()),None)
    if compat is None:
        raise RuntimeError('古いcollision importが見つかりません。互換shimも不在です: '+probe.stderr[-1000:])
    shutil.copy2(compat, WORK/'carry_compat.py')
    original = 'from mujoco_playground._src.collision import geoms_colliding'
    text = joystick_copy.read_text(encoding='utf-8')
    if original not in text:
        raise RuntimeError('joystick.pyのcollision importの書式が予想外です。手動で確認してください')
    joystick_copy.write_text(text.replace(original,'from carry_compat import geoms_colliding'), encoding='utf-8')
    print('旧collision互換shimを作業コピーに適用:',compat)
else:
    print('collision APIはそのまま動作可能')

# 学習時のコードとの差分を明示（互換修正がある場合のみ差分発生）
print('Source SHA256 :',sha256(SNAP_JOYSTICK.read_bytes()).hexdigest())
print('Work   SHA256 :',sha256(joystick_copy.read_bytes()).hexdigest())

旧collision互換shimを作業コピーに適用: /content/drive/MyDrive/OpenDuckRL/delivery_ppo_stage2_150m/continuation_50m_source_snapshot/carry_compat.py
Source SHA256 : 7ac0c1ca9a6df7e5db3bf49e7fa1c5e7057fa607e55c86c7582c8381857d3870
Work   SHA256 : 8461ae1bb6da8be1f4b79fc38aba06ec3713198736eac1f5af428a9961b81605


## 5. 7段階の動画生成スクリプトを保存

Brax `checkpoint.load` で元の正規化統計・PPOパラメータを読み込み、同じ `Joystick(task="flat_terrain")` で8秒間ずつ録画します。**30M・60M・100M付近を追加**し、以前の4段階版で作成した動画は条件が一致すればコピーして再利用します。


In [5]:
%%writefile /content/duck_150m_walk_review/record_walk_150m.py
"""Read-only replay of 150M WALKING PPO Orbax checkpoints.

Run with an isolated working tree copied from the original 150M training snapshot.
Never mutates a checkpoint, source XML, or the carrying/fresh-PPO experiment.
"""
import os
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

from pathlib import Path
import json
import shutil
import subprocess
import sys
import hashlib

import jax
import jax.numpy as jnp
import mujoco
import numpy as np
from brax.training.acme import running_statistics
from brax.training.agents.ppo import checkpoint as brax_checkpoint
from brax.training.agents.ppo import networks as ppo_networks
from mujoco_playground.config import locomotion_params

from playground.open_duck_mini_v2 import joystick

DRIVE = Path('/content/drive/MyDrive/OpenDuckRL')
EXP = DRIVE / 'experiments/ppo_150m_v1'
CKPT_ROOT = EXP / 'checkpoints'
OUTPUT = EXP / 'walk_video_replay_7stages'
LEGACY_OUTPUT = EXP / 'walk_video_replay_4stages'
SCRATCH = Path('/content/duck_150m_walk_video_tmp')

TARGETS = [
    ('early_10.81M', 10_813_440),
    ('early_32.44M', 32_440_320),  # NEW ~30M
    ('earlymid_43.25M', 43_253_760),
    ('middle_64.88M', 64_880_640), # NEW ~60M
    ('middle_75.69M', 75_694_080),
    ('late_97.32M', 97_320_960),   # NEW ~100M
    ('final_151.39M', 151_388_160),
]
LEGACY_STEPS = {10_813_440, 43_253_760, 75_694_080, 151_388_160}

FPS = 15
SECONDS = 8
VX = 0.12
SEED = 1300
WIDTH, HEIGHT = 640, 480
FORCE_OVERWRITE = False


def get_checkpoint(steps):
    matches = sorted(p for p in CKPT_ROOT.glob(f'*_{steps}') if p.is_dir())
    if len(matches) != 1:
        raise FileNotFoundError(f'Need exactly one checkpoint for {steps:,}; got {matches}')
    p = matches[0]
    if not any(p.iterdir()):
        raise RuntimeError(f'Checkpoint directory is empty: {p}')
    return p


def result_rows(folder):
    path = folder / 'replay_results.json'
    if not path.is_file():
        return {}
    try:
        rows = json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(rows, list):
            return {}
        return {int(r['steps']): r for r in rows if isinstance(r, dict) and 'steps' in r}
    except (OSError, ValueError, KeyError, TypeError):
        return {}


def legacy_video_is_compatible(row, steps):
    # Previous 4-stage notebook used the exact same fixed evaluation settings.
    return (
        isinstance(row, dict)
        and row.get('steps') == steps
        and row.get('seed') == SEED
        and abs(float(row.get('vx_command_m_s', -1)) - VX) < 1e-9
        and row.get('requested_video_s') == SECONDS
        and row.get('noise_disabled_for_comparison') is True
        and row.get('push_disabled_for_comparison') is True
        and row.get('PPO_training_performed') is False
    )


def make_env():
    # Use the original walking task. Remove perturbations only for deterministic,
    # controlled comparison; no payload / carry environment is imported.
    config = joystick.default_config()
    try:
        config.noise_config.level = 0.0
        config.push_config.enable = False
    except (AttributeError, KeyError) as exc:
        print('Warning: no noise/push config override:', exc, flush=True)
    env = joystick.Joystick(task='flat_terrain', config=config)
    obs_size = env.observation_size
    action_size = int(env.action_size)
    obs_len = int(obs_size['state'][0]) if isinstance(obs_size, dict) else int(obs_size['state'][0])
    print('Original walking environment dimensions:', obs_len, action_size, flush=True)
    try:
        from playground.open_duck_mini_v2 import constants
        xml_path = constants.task_to_xml('flat_terrain')
        print('Walking XML used:', xml_path, flush=True)
        xml_path = Path(str(xml_path))
        if xml_path.is_file():
            print('Walking XML SHA256:', hashlib.sha256(xml_path.read_bytes()).hexdigest(), flush=True)
    except Exception as exc:
        print('Walking XML provenance could not be printed:', exc, flush=True)
    if (obs_len, action_size) != (101, 14):
        raise RuntimeError(
            f'Mismatch: obs={obs_len}, actions={action_size}; expected (101,14). '
            'Do NOT regenerate video with a different robot. '
            'Restore the 150M original joystick.py and robot XML from '
            'experiments/ppo_150m_v1/source/playground/.'
        )
    if not (float(env.dt) > 0):
        raise RuntimeError('Invalid environment timestep')
    if SECONDS / float(env.dt) > 500:
        raise RuntimeError('Requested video crosses command re-sampling step. '
                           'Reduce SECONDS or pin command in the source-compatible way.')
    return env


def main():
    print('JAX devices:', jax.devices(), flush=True)
    if not shutil.which('ffmpeg'):
        raise RuntimeError('ffmpeg executable missing')
    assert CKPT_ROOT.is_dir(), CKPT_ROOT
    OUTPUT.mkdir(parents=True, exist_ok=True)
    SCRATCH.mkdir(parents=True, exist_ok=True)

    env = make_env()
    model = env.mj_model
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'trunk_assembly')
    if body_id < 0:
        raise RuntimeError('No trunk_assembly body in scene; wrong walking XML?')

    conf = locomotion_params.brax_ppo_config('BerkeleyHumanoidJoystickFlatTerrain')
    nets = ppo_networks.make_ppo_networks(
        observation_size=env.observation_size,
        action_size=env.action_size,
        preprocess_observations_fn=running_statistics.normalize,
        **dict(conf.get('network_factory', {})),
    )
    make_policy = ppo_networks.make_inference_fn(nets)

    @jax.jit
    def act(params, obs, key):
        action, _ = make_policy(params, deterministic=True)(obs, key)
        return action

    reset = jax.jit(env.reset)
    step = jax.jit(env.step)
    command = jnp.array([VX, 0., 0., 0., 0., 0., 0.])
    # The original joystick module exports the exact collision utility it used.
    collides = joystick.geoms_colliding

    frame_steps = np.rint(np.arange(FPS * SECONDS) / FPS / float(env.dt)).astype(int)
    results = []
    legacy_rows = result_rows(LEGACY_OUTPUT)
    already_rows = result_rows(OUTPUT)
    md = mujoco.MjData(model)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.azimuth = 135
    cam.elevation = -18
    cam.distance = 1.35

    with mujoco.Renderer(model, width=WIDTH, height=HEIGHT) as renderer:
        for tag, steps in TARGETS:
            checkpoint_dir = get_checkpoint(steps)
            mp4 = OUTPUT / f'walking_ppo_{tag.replace(".", "p")}.mp4'
            if mp4.is_file() and mp4.stat().st_size > 1024 and not FORCE_OVERWRITE:
                print('Already recorded (skip):', mp4, flush=True)
                prior = dict(already_rows.get(steps, {}))
                prior.update({'stage':tag, 'steps':steps, 'video':str(mp4), 'status':'reused_7stages'})
                results.append(prior)
                continue

            # Existing 4 MP4s can be reused only if their evaluation settings
            # are documented as identical. Never edit files in LEGACY_OUTPUT.
            previous_file = LEGACY_OUTPUT / mp4.name
            previous_row = legacy_rows.get(steps)
            if (not FORCE_OVERWRITE and steps in LEGACY_STEPS
                    and previous_file.is_file() and previous_file.stat().st_size > 1024
                    and legacy_video_is_compatible(previous_row, steps)):
                shutil.copy2(previous_file, mp4)
                previous_row = dict(previous_row)
                previous_row.update({'stage': tag, 'steps': steps, 'video': str(mp4),
                                     'status': 'copied_from_4stages'})
                results.append(previous_row)
                print('REUSED prior 4-stage MP4:', previous_file, '->', mp4, flush=True)
                (OUTPUT / 'replay_results.json').write_text(
                    json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')
                continue

            print(f'\nRestoring: {tag} from {checkpoint_dir}', flush=True)
            params = brax_checkpoint.load(str(checkpoint_dir))
            if not isinstance(params, (tuple,list)) or len(params) != 3:
                raise RuntimeError('Unexpected checkpoint structure; expected '
                                   '(normalizer, policy, value) tuple')

            state = reset(jax.random.PRNGKey(SEED))
            info = dict(state.info)
            info['command'] = command
            contacts = jnp.array([
                collides(state.data, foot, env._floor_geom_id)
                for foot in env._feet_geom_id
            ])
            state = state.replace(info=info,
                                  obs=env._get_obs(state.data, info, contacts))

            rng = jax.random.PRNGKey(5300)
            try:
                rng, akey = jax.random.split(rng)
                action = act(params, state.obs, akey)
                actual_action_size = int(np.asarray(action).size)
                assert actual_action_size == int(env.action_size), (actual_action_size, env.action_size)
                print(f'Checkpoint restored and policy tested OK ({actual_action_size} actions).', flush=True)
            except Exception:
                print('Policy inference failed. Original joystick.py / network config '
                      'and checkpoint format must match; DO NOT retrain just for this error.', flush=True)
                raise

            q0 = np.asarray(state.data.qpos).copy()
            root_addr = int(env._floating_base_qpos_addr)
            pos0 = q0[root_addr:root_addr + 3].copy()
            sim_step = 0
            ended = False
            done_at_s = None
            video_last = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
            temporary = SCRATCH / mp4.name
            temporary.unlink(missing_ok=True)

            ff = subprocess.Popen(
                ['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
                 '-f', 'rawvideo', '-pixel_format', 'rgb24',
                 '-video_size', f'{WIDTH}x{HEIGHT}', '-framerate', str(FPS),
                 '-i', 'pipe:0', '-an', '-c:v', 'libx264', '-preset', 'veryfast',
                 '-crf', '23', '-pix_fmt', 'yuv420p', str(temporary)],
                stdin=subprocess.PIPE, stderr=subprocess.PIPE,
            )
            try:
                for frame_index, goal in enumerate(frame_steps):
                    while sim_step < goal and not ended:
                        rng, akey = jax.random.split(rng)
                        action = act(params, state.obs, akey)
                        state = step(state, action)
                        sim_step += 1
                        pos = np.asarray(state.data.qpos)
                        vel = np.asarray(state.data.qvel)
                        if not (np.isfinite(pos).all() and np.isfinite(vel).all()):
                            ended = True
                            print('Nonfinite simulation values at step', sim_step, flush=True)
                            break
                        if bool(np.asarray(state.done)):
                            ended = True
                            done_at_s = round(sim_step * float(env.dt), 3)
                            print('Terminated / fell at:', done_at_s, 's', flush=True)

                    if np.isfinite(np.asarray(state.data.qpos)).all() and np.isfinite(np.asarray(state.data.qvel)).all():
                        md.qpos[:] = np.asarray(state.data.qpos)
                        md.qvel[:] = np.asarray(state.data.qvel)
                        mujoco.mj_forward(model, md)
                        cam.lookat[:] = np.asarray(md.xpos[body_id])
                        cam.lookat[2] = 0.32
                        renderer.update_scene(md, camera=cam)
                        video_last = renderer.render().copy()
                    ff.stdin.write(video_last.tobytes())
                    if frame_index % 30 == 0:
                        print('frames:', frame_index, '/', len(frame_steps), flush=True)
            except BaseException:
                ff.kill()
                ff.wait()
                temporary.unlink(missing_ok=True)
                raise
            finally:
                if ff.stdin is not None and not ff.stdin.closed:
                    ff.stdin.close()
            err = ff.stderr.read().decode('utf-8', 'replace')
            ff_code = ff.wait()
            if ff_code:
                temporary.unlink(missing_ok=True)
                raise RuntimeError(f'ffmpeg failed code={ff_code}: {err[-2000:]}')
            if not temporary.is_file() or temporary.stat().st_size < 1024:
                raise RuntimeError('No usable mp4 generated: ' + str(temporary))
            shutil.copy2(temporary, mp4)
            temporary.unlink()

            pos1 = np.asarray(state.data.qpos)[root_addr:root_addr + 3].copy()
            row = {
                'stage':tag, 'steps':steps, 'checkpoint':str(checkpoint_dir),
                'video':str(mp4), 'seed':SEED, 'vx_command_m_s':VX,
                'requested_video_s':SECONDS, 'simulated_s':round(sim_step*float(env.dt),3),
                'fell_at_s':done_at_s,
                'movement_xy_m':round(float(np.linalg.norm(pos1[:2]-pos0[:2])),4),
                'root_dxyz_m':np.round(pos1-pos0,5).tolist(),
                'noise_disabled_for_comparison':True,
                'push_disabled_for_comparison':True,
                'PPO_training_performed':False,
            }
            results.append(row)
            print('SAVED:', mp4, '\nRESULT:', json.dumps(row,ensure_ascii=False), flush=True)
            (OUTPUT / 'replay_results.json').write_text(
                json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')

    (OUTPUT / 'replay_results.json').write_text(json.dumps(results,indent=2,ensure_ascii=False),encoding='utf-8')
    print('\nCOMPLETED. Original PPO training was not changed.', flush=True)


if __name__ == '__main__':
    main()


Writing /content/duck_150m_walk_review/record_walk_150m.py


## 6. 復元・録画を実行（GPU推奨）

7段階の動画を新しい出力先へ保存します。旧4段階版のMP4は、元の録画条件がレポートで確認できる場合は**コピーして再利用**します。残りだけOrbax重みから再生・録画するため、**再学習は不要**です。


In [6]:
from pathlib import Path
import os, subprocess
SCRIPT = WORK/'record_walk_150m.py'
assert SCRIPT.is_file(), SCRIPT
ENV = os.environ.copy()
ENV['MUJOCO_GL'] = 'egl'
ENV['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
ENV['PYTHONPATH'] = str(WORK) + os.pathsep + ENV.get('PYTHONPATH','')
print('復元対象: 10.81M / 32.44M / 43.25M / 64.88M / 75.69M / 97.32M / 151.39M')
print('学習操作はありません。動画のみ生成します。')
subprocess.run([str(PYTHON),'-u',str(SCRIPT)], cwd=WORK, env=ENV, check=True)
print('7段階の録画を確認してください。')

復元対象: 10.81M / 32.44M / 43.25M / 64.88M / 75.69M / 97.32M / 151.39M
学習操作はありません。動画のみ生成します。
7段階の録画を確認してください。


## 7. 保存された7本をColab上で再生

動画はステップ数の昇順で表示します。**75.69M前後の変化**を特に見たい場合は、64.88M・75.69M・97.32Mを見比べてください。


In [7]:
from pathlib import Path
from IPython.display import display, Markdown, Video
import json
OUT = EXP / 'walk_video_replay_7stages'
for title, basename in [
    ('約10M / 10.81M','walking_ppo_early_10p81M.mp4'),
    ('約30M / 32.44M【追加】','walking_ppo_early_32p44M.mp4'),
    ('約40M / 43.25M','walking_ppo_earlymid_43p25M.mp4'),
    ('約60M / 64.88M【追加】','walking_ppo_middle_64p88M.mp4'),
    ('約75M / 75.69M','walking_ppo_middle_75p69M.mp4'),
    ('約100M / 97.32M【追加】','walking_ppo_late_97p32M.mp4'),
    ('約151M / 151.39M','walking_ppo_final_151p39M.mp4'),
]:
    path = OUT/basename
    display(Markdown(f'### {title}'))
    if path.is_file():
        print(path)
        display(Video(str(path), embed=True, width=720))
    else:
        print('未作成:',path)
summary = OUT/'replay_results.json'
if summary.is_file():
    print('\n=== 比較結果（横移動距離・転倒時刻）===')
    for row in json.loads(summary.read_text(encoding='utf-8')):
        print(row.get('stage'),
              'fall(s)=',row.get('fell_at_s'),
              'xy distance(m)=',row.get('movement_xy_m'),
              'status=',row.get('status','new'))

### 約10M / 10.81M

/content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/walk_video_replay_7stages/walking_ppo_early_10p81M.mp4


### 約30M / 32.44M【追加】

/content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/walk_video_replay_7stages/walking_ppo_early_32p44M.mp4


### 約40M / 43.25M

...

### 約100M / 97.32M【追加】

/content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/walk_video_replay_7stages/walking_ppo_late_97p32M.mp4


### 約151M / 151.39M

/content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/walk_video_replay_7stages/walking_ppo_final_151p39M.mp4



=== 比較結果（横移動距離・転倒時刻）===
early_10.81M fall(s)= None xy distance(m)= 0.0272 status= copied_from_4stages
early_32.44M fall(s)= None xy distance(m)= 0.0398 status= new
earlymid_43.25M fall(s)= None xy distance(m)= 0.0445 status= copied_from_4stages
middle_64.88M fall(s)= None xy distance(m)= 0.566 status= new
middle_75.69M fall(s)= None xy distance(m)= 0.7165 status= copied_from_4stages
late_97.32M fall(s)= None xy distance(m)= 0.7819 status= new
final_151.39M fall(s)= None xy distance(m)= 0.6875 status= copied_from_4stages


## よくあるエラー

- **重みはあるが `checkpoint.load()` が失敗する**：Orbax形式やライブラリのバージョン差が疑われます。PPOをやり直さず、エラー全文と `orbax.__version__`、`brax.__version__` を確認してください。
- **観測101 / 行動14と一致しない**：学習当時と異なるソースやXMLを読み込んでいます。表示がおかしい動画を作らず停止します。`experiments/ppo_150m_v1/source/` 内のファイルを確認してください。
- **JAX/CUDAのエラー**：ColabのGPUランタイムを選択し、Python環境セルから再実行してください。
- **`mujoco.FatalError`/EGLエラー**：レンダリング環境の問題です。学習済み重みが失われたわけではありません。

**今回の7本が歩行できていれば、運搬用PPOの初期姿勢やトレイ位置の問題を切り分けられます。** うまく再生できなかった場合は、再学習を始めずにエラーログを送ってください。

In [8]:
from pathlib import Path

OUT = Path(
    "/content/drive/MyDrive/OpenDuckRL/"
    "experiments/ppo_150m_v1/walk_video_replay_7stages"
)

names = {
    "10M":  "walking_ppo_early_10p81M.mp4",
    "30M":  "walking_ppo_early_32p44M.mp4",
    "40M":  "walking_ppo_earlymid_43p25M.mp4",
    "60M":  "walking_ppo_middle_64p88M.mp4",
    "75M":  "walking_ppo_middle_75p69M.mp4",
    "100M": "walking_ppo_late_97p32M.mp4",
    "151M": "walking_ppo_final_151p39M.mp4",
}

print("保存先:", OUT)
print("=" * 55)

saved = 0

for stage, filename in names.items():
    path = OUT / filename

    if path.is_file() and path.stat().st_size > 1024:
        size = path.stat().st_size / 1024**2
        print(f"✅ {stage:>4} : 保存済み ({size:.2f} MB)")
        saved += 1
    else:
        print(f"❌ {stage:>4} : 未保存")

print("=" * 55)
print(f"保存済み: {saved} / 7 本")

if saved == 7:
    print("✅ 全7段階の動画ファイルを確認しました！")
else:
    print("不足分は動画生成セルを再実行してください。")

保存先: /content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/walk_video_replay_7stages
✅  10M : 保存済み (0.13 MB)
✅  30M : 保存済み (0.11 MB)
✅  40M : 保存済み (0.09 MB)
✅  60M : 保存済み (0.17 MB)
✅  75M : 保存済み (0.19 MB)
✅ 100M : 保存済み (0.19 MB)
✅ 151M : 保存済み (0.19 MB)
保存済み: 7 / 7 本
✅ 全7段階の動画ファイルを確認しました！
